In [1]:
"""
Import librairies
"""
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
from tensorflow.keras import backend as K
import numpy as np
import matplotlib.pyplot as plt #can be replace by plotly
from pathlib import Path
import gc

In [2]:
"""
Hyperparameters
"""
train_split=0.75
test_split=0.25
epoch=5
lambda_2=0.001
learning_rate=0.01

In [3]:
"""
Variables
"""
BATCH_SIZE = 32
dossier_racine = Path(f"../data/processing/")
size=64

In [4]:
result=[]

In [14]:
img_size = (size, size)#transform l'image pour qu'elle soit lisible par tensorflow

# Charger le dataset complet non-batché pour un split précis
full_dataset = tf.keras.utils.image_dataset_from_directory(
    dossier_racine,
    labels='inferred',
    label_mode='int',
    class_names=['False', 'True'],
    image_size=img_size,
    batch_size=32, # Important : on gère les batchs après le split
    shuffle=True,
    seed=42
)
dataset_size = len(full_dataset)
train_size = int(train_split * dataset_size)
test_size = int(test_split * dataset_size)

train_dataset = full_dataset.take(train_size)
test_val_dataset = full_dataset.skip(train_size)
test_dataset = test_val_dataset.take(test_size)

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

Found 41398 files belonging to 2 classes.


In [6]:
"""
Add callback
"""
early_stopping_callback = EarlyStopping(
monitor='val_loss', # The metric to monitor for improvement
patience=8,         # Number of epochs with no improvement after which training will be stopped
verbose=1,          # Show a message when stopping
restore_best_weights=True # Restores model weights from the epoch with the best value of the monitored quantity.
)

In [15]:
"""
Load and configure model
"""

model_layers = [
    keras.Input(shape=(None, None, 3)), 
    
    # Redimensionnement 
    layers.Resizing(size, size),

    #data augmentation
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),

    #Scale to 0 to 1
    layers.Rescaling(1./255),
    
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.1),
]

if size == 256: 
    model_layers.extend([
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.1),
    ])

model_layers.extend([
    layers.Flatten(),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(lambda_2)),
    layers.Dense(1, activation='sigmoid', kernel_regularizer=regularizers.l2(lambda_2))
])

model = keras.Sequential(model_layers)
print(model.summary())


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing_1 (Resizing)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 64, 64, 3)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,625,281 (6.20 MB)

 Trainable params: 1,625,281 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

None


In [16]:
"""
Configure Optimizer
"""
model.compile(optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy'])

In [17]:
"""
Launch model
"""
all_callbacks = [early_stopping_callback]
print(f"Lancement du train du modele pour {size}x{size}")
history = model.fit(
    train_dataset, 
    epochs=epoch, 
    callbacks=all_callbacks, 
    validation_data=test_dataset
)

# Evaluate the model on the unseen test data
test_loss, test_accuracy = model.evaluate(test_dataset)
data={"loss":test_loss,"accuracy":test_accuracy}
print(f"\nTest Accuracy: {test_accuracy}")
result.append(data)

Lancement du train du modele pour 64x64
Epoch 1/5
970/970 ━━━━━━━━━━━━━━━━━━━━ 34s 34ms/step - accuracy: 0.8058 - loss: 0.3963 - val_accuracy: 0.8268 - val_loss: 0.3495
Epoch 2/5
970/970 ━━━━━━━━━━━━━━━━━━━━ 34s 35ms/step - accuracy: 0.8447 - loss: 0.3383 - val_accuracy: 0.8511 - val_loss: 0.3332
Epoch 3/5
970/970 ━━━━━━━━━━━━━━━━━━━━ 34s 35ms/step - accuracy: 0.8591 - loss: 0.3223 - val_accuracy: 0.8407 - val_loss: 0.3599
Epoch 4/5
970/970 ━━━━━━━━━━━━━━━━━━━━ 36s 37ms/step - accuracy: 0.8657 - loss: 0.3127 - val_accuracy: 0.8725 - val_loss: 0.2985
Epoch 5/5
970/970 ━━━━━━━━━━━━━━━━━━━━ 33s 34ms/step - accuracy: 0.8699 - loss: 0.3028 - val_accuracy: 0.8611 - val_loss: 0.3146
Restoring model weights from the end of the best epoch: 4.
323/323 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.8726 - loss: 0.2985

Test Accuracy: 0.8725812435150146


In [ ]:
model.save(f"Modele_cnn_{size}x{size}_{epoch}max.keras")

In [10]:
import os
import tensorflow as tf
from pathlib import Path

# Target your specific dataset path
dossier_racine = Path("../data/processing/")

# Iterate through all files in the directory and subdirectories
for filepath in dossier_racine.rglob('*'):
    if filepath.is_file():
        try:
            # Attempt to read and decode the image exactly as the dataset pipeline does
            image_contents = tf.io.read_file(str(filepath))
            _ = tf.image.decode_image(image_contents, expand_animations=False)
        except tf.errors.InvalidArgumentError:
            print(f"Corrupted image detected and deleted: {filepath}")
            os.remove(filepath) # Deletes the bad file automatically
        except Exception as e:
            print(f"Other issue with file {filepath}: {e}")

Corrupted image detected and deleted: ..\data\processing\False\Painting_painting_02662.jpg
Corrupted image detected and deleted: ..\data\processing\False\Schematics_desktop.ini


In [12]:
import os
import unicodedata

def enlever_accents(chaine):
    """Supprime les accents d'une chaîne de caractères."""
    chaine_normalisee = unicodedata.normalize('NFKD', chaine)
    return "".join([c for c in chaine_normalisee if not unicodedata.combining(c)])

# Chemin de votre dossier racine
dossier_racine = "../data/processing/"

# os.walk avec topdown=False permet de renommer les fichiers à l'intérieur 
# des dossiers AVANT de renommer les dossiers eux-mêmes
for root, dossiers, fichiers in os.walk(dossier_racine, topdown=False):
    
    # 1. Renommer les fichiers
    for nom_fichier in fichiers:
        nom_propre = enlever_accents(nom_fichier)
        if nom_fichier != nom_propre:
            ancien_chemin = os.path.join(root, nom_fichier)
            nouveau_chemin = os.path.join(root, nom_propre)
            os.rename(ancien_chemin, nouveau_chemin)
            print(f"Fichier renommé : {nom_fichier} -> {nom_propre}")
            
    # 2. Renommer les dossiers
    for nom_dossier in dossiers:
        nom_propre = enlever_accents(nom_dossier)
        if nom_dossier != nom_propre:
            ancien_chemin = os.path.join(root, nom_dossier)
            nouveau_chemin = os.path.join(root, nom_propre)
            os.rename(ancien_chemin, nouveau_chemin)
            print(f"Dossier renommé : {nom_dossier} -> {nom_propre}")

print("Nettoyage des accents terminé !")

Nettoyage des accents terminé !
